<a href="https://colab.research.google.com/github/mtzwyu/TTNT/blob/main/ToMau/BTVN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import random

# --- THÔNG SỐ GIẢ LẬP  ---
CLASSES = ['10A1', '10A2']
SUBJECTS = ['Toan', 'Tin']
TEACHERS = {'Toan': ['GV_Toan1', 'GV_Toan2'], 'Tin': ['GV_Tin1']}
TIME_SLOTS = [f"Thu_{day}_Ca_{ca}" for day in range(2, 7) for ca in [1, 2]] # 10 khung giờ

# Yêu cầu giảng dạy: Mỗi lớp cần học 2 tiết Toán, 1 tiết Tin trong tuần
TEACHING_REQUIREMENTS = [
    ('10A1', 'Toan'), ('10A1', 'Toan'), ('10A1', 'Tin'),
    ('10A2', 'Toan'), ('10A2', 'Toan'), ('10A2', 'Tin')
]

# --- THUẬT TOÁN DI TRUYỀN (GA) ---
class Schedule:
    def __init__(self):
        # Genotype: Danh sách các phân công: (Lớp, Môn, Giáo Viên, Khung giờ)
        self.assignments = []
        self.fitness = 0

    def random_init(self):
        for req in TEACHING_REQUIREMENTS:
            class_name, subject = req
            teacher = random.choice(TEACHERS[subject])
            time_slot = random.choice(TIME_SLOTS)
            self.assignments.append((class_name, subject, teacher, time_slot))

    def calculate_fitness(self):
        conflicts = 0
        # Ràng buộc: Giáo viên không thể dạy 2 lớp cùng 1 lúc, Lớp không học 2 môn cùng 1 lúc
        for i in range(len(self.assignments)):
            for j in range(i + 1, len(self.assignments)):
                c1, s1, t1, time1 = self.assignments[i]
                c2, s2, t2, time2 = self.assignments[j]

                if time1 == time2:
                    if t1 == t2: # Trùng giáo viên
                        conflicts += 1
                    if c1 == c2: # Trùng lớp
                        conflicts += 1

        # Hàm mục tiêu: Thích nghi cao nhất khi không có xung đột
        self.fitness = 1.0 / (1.0 + conflicts)

def crossover(parent1, parent2):
    """Lai ghép 1 điểm cắt"""
    child = Schedule()
    crossover_point = random.randint(1, len(TEACHING_REQUIREMENTS) - 1)
    child.assignments = parent1.assignments[:crossover_point] + parent2.assignments[crossover_point:]
    return child

def mutate(schedule, mutation_rate=0.1):
    """Đột biến: Thay đổi ngẫu nhiên khung giờ hoặc giáo viên của 1 tiết học"""
    if random.random() < mutation_rate:
        idx = random.randint(0, len(schedule.assignments) - 1)
        c, s, t, time = schedule.assignments[idx]

        if random.random() < 0.5:
            # Đổi giáo viên khác cùng chuyên môn
            t = random.choice(TEACHERS[s])
        else:
            # Đổi thời gian
            time = random.choice(TIME_SLOTS)

        schedule.assignments[idx] = (c, s, t, time)

def genetic_algorithm(population_size=50, generations=100):
    # Khởi tạo quần thể
    population = []
    for _ in range(population_size):
        sch = Schedule()
        sch.random_init()
        sch.calculate_fitness()
        population.append(sch)

    for gen in range(generations):
        # Sắp xếp theo độ thích nghi giảm dần
        population.sort(key=lambda x: x.fitness, reverse=True)

        # Nếu tìm được giải pháp tối ưu (không xung đột)
        if population[0].fitness == 1.0:
            print(f"Đã tìm thấy lịch học tối ưu ở thế hệ {gen}!")
            return population[0]

        # Chọn lọc và lai ghép (giữ lại 20% cá thể tốt nhất)
        next_gen = population[:int(0.2 * population_size)]

        while len(next_gen) < population_size:
            p1 = random.choice(population[:25]) # Chọn trong top 50%
            p2 = random.choice(population[:25])
            child = crossover(p1, p2)
            mutate(child)
            child.calculate_fitness()
            next_gen.append(child)

        population = next_gen

    population.sort(key=lambda x: x.fitness, reverse=True)
    return population[0]

# --- THỰC THI CHƯƠNG TRÌNH ---
if __name__ == "__main__":
    print("ĐANG TÌM LỊCH DẠY BẰNG GIẢI THUẬT DI TRUYỀN...")
    best_schedule = genetic_algorithm(population_size=100, generations=200)

    print("\nKẾT QUẢ SẮP LỊCH (Độ thích nghi: {:.2f}):".format(best_schedule.fitness))
    if best_schedule.fitness < 1.0:
        print("Lưu ý: Lịch học vẫn còn xung đột do chưa hội tụ hoàn toàn. Thử tăng số generation hoặc population_size.")
    else:
        print("Lịch học hợp lệ (Không có giáo viên/lớp bị trùng giờ).")

    # Sắp xếp lại danh sách để in ra dễ nhìn theo Lớp và Thời gian
    sorted_assignments = sorted(best_schedule.assignments, key=lambda x: (x[0], x[3]))

    print("-" * 50)
    print(f"{'LỚP':<6} | {'MÔN':<5} | {'GIÁO VIÊN':<10} | {'THỜI GIAN'}")
    print("-" * 50)
    for c, s, t, time in sorted_assignments:
        print(f"{c:<6} | {s:<5} | {t:<10} | {time}")

ĐANG TÌM LỊCH DẠY BẰNG GIẢI THUẬT DI TRUYỀN...
Đã tìm thấy lịch học tối ưu ở thế hệ 0!

KẾT QUẢ SẮP LỊCH (Độ thích nghi: 1.00):
Lịch học hợp lệ (Không có giáo viên/lớp bị trùng giờ).
--------------------------------------------------
LỚP    | MÔN   | GIÁO VIÊN  | THỜI GIAN
--------------------------------------------------
10A1   | Toan  | GV_Toan2   | Thu_4_Ca_1
10A1   | Toan  | GV_Toan1   | Thu_5_Ca_1
10A1   | Tin   | GV_Tin1    | Thu_5_Ca_2
10A2   | Toan  | GV_Toan2   | Thu_2_Ca_1
10A2   | Toan  | GV_Toan2   | Thu_5_Ca_2
10A2   | Tin   | GV_Tin1    | Thu_6_Ca_1
